In [6]:
import os
import pandas as pd
import torch

from tqdm import tqdm

from c3ppi.ppi_predictor.models.C3BCA.embedder.prost5 import ProteinEmbeddingService

/home/c3biolab/c3biolab_projects/doctorals/DPK/c3ppi/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:

id_mapping_1 = "./data/idmapping_1.tsv.gz"
id_mapping_2 = "./data/idmapping_2.tsv.gz"

In [8]:
# Read and combine
df1 = pd.read_csv(id_mapping_1, sep="\t", header=0)
df2 = pd.read_csv(id_mapping_2, sep="\t", header=0)

# Append
df = pd.concat([df1, df2], axis=0)
df

,From,Entry,Reviewed,Entry Name,Protein names,Gene Names,Organism,Length,Sequence
0,Q9ZN16,Q9ZN16,unreviewed,Q9ZN16_HELPJ,DNA (cytosine-5-)-methyltransferase (EC 2.1.1.37),jhp_0044,Helicobacter pylori (strain J99 / ATCC 700824)...,185.0,MLVSRFLNAIDPFNLGVLLSRFQIKNGCIYGVCSYKVSKFTPGYEE...
1,P39683,P39683,reviewed,NPT1_YEAST,Nicotinate phosphoribosyltransferase (NAPRTase...,NPT1 YOR209C,Saccharomyces cerevisiae (strain ATCC 204508 /...,429.0,MSEPVIKSLLDTDMYKITMHAAVFTNFPDVTVTYKYTNRSSQLTFN...
2,Q9CPR1,Q9CPR1,reviewed,RWDD4_MOUSE,RWD domain-containing protein 4,Rwdd4 Rwdd4a,Mus musculus (Mouse),188.0,MGANEDQEMELEALRSIYEGDNSFRELSPVSFQYRIGEDGDPKAFL...
3,P51911,P51911,reviewed,CNN1_HUMAN,"Calponin-1 (Basic calponin) (Calponin H1, smoo...",CNN1,Homo sapiens (Human),297.0,MSSAHFNRGPAYGLSAEVKNKLAQKYDHQREQELREWIEGVTGRRI...
4,O04314,O04314,reviewed,JAL30_ARATH,PYK10-binding protein 1 (Jacalin-related lecti...,PBP1 JAL30 JIP PBPI At3g16420 MDC8.5 T02O04.8,Arabidopsis thaliana (Mouse-ear cress),298.0,MAQKVEAQGGKGANLWDDGSTHDAVTKIQLAAGIDGIQYVQFDYVK...
...,...,...,...,...,...,...,...,...,...
89995,Q63470,Q63470,reviewed,DYR1A_RAT,Dual specificity tyrosine-phosphorylation-regu...,Dyrk1a Dyrk,Rattus norvegicus (Rat),763.0,MHTGGETSACKPSSVRLAPSFSFHAAGLQMAAQMPHSHQYSDRRQP...
89996,P0A772,P0A772,reviewed,NRDI_ECOLI,Protein NrdI,nrdI ygaO b2674 JW2649,Escherichia coli (strain K12),136.0,MSQLVYFSSSSENTQRFIERLGLPAVRIPLNERERIQVDEPYILIV...
89997,P23524,P23524,reviewed,GLXK1_ECOLI,Glycerate 2-kinase (EC 2.7.1.165) (Glycerate k...,garK yhaD b3124 JW3093,Escherichia coli (strain K12),381.0,MKIVIAPDSYKESLSASEVAQAIEKGFREIFPDAQYVSVPVADGGE...
89998,C9JVN1,C9JVN1,unreviewed,C9JVN1_HUMAN,Melanotransferrin,MELTF,Homo sapiens (Human),181.0,MRGPSGALWLLLALRTVLGGMEVRWCATSDPEQHKCGNMSEAFREA...


In [9]:
# Drop na within Sequence
df = df.dropna(subset=["Sequence"])
df

,From,Entry,Reviewed,Entry Name,Protein names,Gene Names,Organism,Length,Sequence
0,Q9ZN16,Q9ZN16,unreviewed,Q9ZN16_HELPJ,DNA (cytosine-5-)-methyltransferase (EC 2.1.1.37),jhp_0044,Helicobacter pylori (strain J99 / ATCC 700824)...,185.0,MLVSRFLNAIDPFNLGVLLSRFQIKNGCIYGVCSYKVSKFTPGYEE...
1,P39683,P39683,reviewed,NPT1_YEAST,Nicotinate phosphoribosyltransferase (NAPRTase...,NPT1 YOR209C,Saccharomyces cerevisiae (strain ATCC 204508 /...,429.0,MSEPVIKSLLDTDMYKITMHAAVFTNFPDVTVTYKYTNRSSQLTFN...
2,Q9CPR1,Q9CPR1,reviewed,RWDD4_MOUSE,RWD domain-containing protein 4,Rwdd4 Rwdd4a,Mus musculus (Mouse),188.0,MGANEDQEMELEALRSIYEGDNSFRELSPVSFQYRIGEDGDPKAFL...
3,P51911,P51911,reviewed,CNN1_HUMAN,"Calponin-1 (Basic calponin) (Calponin H1, smoo...",CNN1,Homo sapiens (Human),297.0,MSSAHFNRGPAYGLSAEVKNKLAQKYDHQREQELREWIEGVTGRRI...
4,O04314,O04314,reviewed,JAL30_ARATH,PYK10-binding protein 1 (Jacalin-related lecti...,PBP1 JAL30 JIP PBPI At3g16420 MDC8.5 T02O04.8,Arabidopsis thaliana (Mouse-ear cress),298.0,MAQKVEAQGGKGANLWDDGSTHDAVTKIQLAAGIDGIQYVQFDYVK...
...,...,...,...,...,...,...,...,...,...
89995,Q63470,Q63470,reviewed,DYR1A_RAT,Dual specificity tyrosine-phosphorylation-regu...,Dyrk1a Dyrk,Rattus norvegicus (Rat),763.0,MHTGGETSACKPSSVRLAPSFSFHAAGLQMAAQMPHSHQYSDRRQP...
89996,P0A772,P0A772,reviewed,NRDI_ECOLI,Protein NrdI,nrdI ygaO b2674 JW2649,Escherichia coli (strain K12),136.0,MSQLVYFSSSSENTQRFIERLGLPAVRIPLNERERIQVDEPYILIV...
89997,P23524,P23524,reviewed,GLXK1_ECOLI,Glycerate 2-kinase (EC 2.7.1.165) (Glycerate k...,garK yhaD b3124 JW3093,Escherichia coli (strain K12),381.0,MKIVIAPDSYKESLSASEVAQAIEKGFREIFPDAQYVSVPVADGGE...
89998,C9JVN1,C9JVN1,unreviewed,C9JVN1_HUMAN,Melanotransferrin,MELTF,Homo sapiens (Human),181.0,MRGPSGALWLLLALRTVLGGMEVRWCATSDPEQHKCGNMSEAFREA...


In [10]:

# Drop duplicates
df = df.drop_duplicates(subset=["Entry"])
df = df.reset_index(drop=True)

# Save to csv
df.to_csv("./data/proteins.csv", index=False)

In [17]:
emb_dir = "./data/embeddings"
if not os.path.exists(emb_dir):
    os.makedirs(emb_dir)

In [12]:
service = ProteinEmbeddingService()

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


In [18]:
proteins = list(zip(df["Entry"], df["Sequence"]))

In [19]:

for p in tqdm(proteins):

    emb_file = os.path.join(emb_dir, f'{p[0]}_embedding.pt')

    if os.path.isfile(emb_file):
        continue
    
    try:
        embedding = service.embed_single_sequence(p[1])
        embedding_tensor = torch.tensor(embedding)
        torch.save(embedding_tensor, emb_file)

    except Exception as e:
        print(f"Failed to embed {p[0]}: {e}")

  0%|          | 0/113865 [00:00<?, ?it/s]/tmp/ipykernel_1656499/886353807.py:10: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  embedding_tensor = torch.tensor(embedding)
 27%|██▋       | 30626/113865 [45:03<1:25:07, 16.30it/s] 

Failed to embed A0A0C4DG59: CUDA out of memory. Tried to allocate 76.82 GiB. GPU 0 has a total capacity of 7.78 GiB of which 3.11 GiB is free. Including non-PyTorch memory, this process has 4.23 GiB memory in use. Of the allocated memory 3.22 GiB is allocated by PyTorch, and 833.29 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)


 38%|███▊      | 43063/113865 [1:03:01<1:43:21, 11.42it/s]

Failed to embed A0A1B0GXE3: CUDA out of memory. Tried to allocate 77.00 GiB. GPU 0 has a total capacity of 7.78 GiB of which 3.41 GiB is free. Including non-PyTorch memory, this process has 3.90 GiB memory in use. Of the allocated memory 3.22 GiB is allocated by PyTorch, and 498.15 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)


 85%|████████▌ | 97306/113865 [2:22:32<20:25, 13.51it/s]  

Failed to embed Q8WZ42: CUDA out of memory. Tried to allocate 70.34 GiB. GPU 0 has a total capacity of 7.78 GiB of which 3.58 GiB is free. Including non-PyTorch memory, this process has 3.70 GiB memory in use. Of the allocated memory 3.18 GiB is allocated by PyTorch, and 335.69 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)


100%|██████████| 113865/113865 [2:46:33<00:00, 11.39it/s] 
